In [ ]:
import pprint

In [1]:
from litellm import completion

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "The unit of temperature.",
                    },
                },
                "required": ["location"],
            }
        }
    }
]

messages = [{"role": "user", "content": "What's the weather like in Boston today?"}]

response = completion(
    model="openai/unsloth/gemma-4-E4B-it-GGUF:Q8_0",  # name can be anything locally
    api_base="http://localhost:8080/v1",
    api_key="not-needed",   # llama.cpp doesn't require it
    messages=messages,
    tools=tools,
)

print(response.choices[0].message.tool_calls[0].function)


Function(arguments='{"location":"Boston, MA"}', name='get_current_weather')


In [ ]:
!pip install llama_index
!pip install llama_index.llms.litellm

In [29]:
import nest_asyncio
from llama_index.llms.litellm import LiteLLM
from llama_index.core.agent.workflow import ReActAgent, AgentStream
from llama_index.core.tools import FunctionTool

nest_asyncio.apply()

def add(a:int|float, b:int|float) -> int|float:
    """Adds 2 integers or floats"""
    return a+b

llm = LiteLLM(
    model="openai/unsloth/gemma-4-E4B-it-GGUF:Q8_0",
    api_base="http://localhost:8080/v1",
    api_key="sk-no-key-required"
)

agent = ReActAgent(
    tools=[FunctionTool.from_defaults(add)],
    llm=llm
)
handler = agent.run("What is 21.921 + 23.21?")
async for event in handler.stream_events():
    if isinstance(event, AgentStream):
        print(event.delta, end="", flush=True)

Thought: The user is asking for the sum of two numbers, 21.921 and 23.21. I have an 'add' tool that can perform addition. I will use this tool.
Action: add
Action Input: {"a": 21.921, "b": 23.21}Thought: I have used the 'add' tool and received the result, 45.131. This is the answer to the user's question.
Answer: 45.131